---
# Lab 1

In [15]:
!pip install pyspark

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, MapType
from pyspark.sql.window import Window

# Create a SparkSession – the entry point for all PySpark operations
# .master("local[*]") runs locally using all available CPU cores
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("PySpark 100 Functions Lab") \
    .getOrCreate()

# Suppress verbose INFO logs for cleaner output
spark.sparkContext.setLogLevel("ERROR")

print("✅ SparkSession created successfully!")
print(f"   Spark version: {spark.version}")

✅ SparkSession created successfully!
   Spark version: 4.0.2


## Section 1 — Practice Exercises

In [17]:
students = spark.createDataFrame([
    (101, "Ali",     "CS",  88),
    (102, "Mona",    "CS",  95),
    (103, "Youssef", "IT",  72),
    (104, "Nour",    "IT",  81),
    (105, "Sara",    "AI",  91),
    (106, "Omar",    "AI",  85)
], ["student_id", "name", "major", "score"])
students.show()

+----------+-------+-----+-----+
|student_id|   name|major|score|
+----------+-------+-----+-----+
|       101|    Ali|   CS|   88|
|       102|   Mona|   CS|   95|
|       103|Youssef|   IT|   72|
|       104|   Nour|   IT|   81|
|       105|   Sara|   AI|   91|
|       106|   Omar|   AI|   85|
+----------+-------+-----+-----+



### Exercise 1 — Filter AI students with score > 90

In [18]:
result = students.filter((F.col("major") == "AI") & (F.col("score") > 90))
result.show()

+----------+----+-----+-----+
|student_id|name|major|score|
+----------+----+-----+-----+
|       105|Sara|   AI|   91|
+----------+----+-----+-----+



### Exercise 2 — Add `result` column (Pass / Fail)

In [19]:
result = students.withColumn(
    "result",
    F.when(F.col("score") >= 80, "Pass").otherwise("Fail")
)
result.show()

+----------+-------+-----+-----+------+
|student_id|   name|major|score|result|
+----------+-------+-----+-----+------+
|       101|    Ali|   CS|   88|  Pass|
|       102|   Mona|   CS|   95|  Pass|
|       103|Youssef|   IT|   72|  Fail|
|       104|   Nour|   IT|   81|  Pass|
|       105|   Sara|   AI|   91|  Pass|
|       106|   Omar|   AI|   85|  Pass|
+----------+-------+-----+-----+------+



### Exercise 3 — Average score per major (sorted highest → lowest)

In [20]:
result = (students
    .groupBy("major")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
result.show()

+-----+---------+
|major|avg_score|
+-----+---------+
|   CS|     91.5|
|   AI|     88.0|
|   IT|     76.5|
+-----+---------+



### Exercise 4 — Count distinct majors

In [21]:
result = students.select(F.countDistinct("major").alias("distinct_majors"))
result.show()

+---------------+
|distinct_majors|
+---------------+
|              3|
+---------------+



### Exercise 5 — Rename `major` → `department`, drop `student_id`

In [22]:
result = students.withColumnRenamed("major", "department").drop("student_id")
result.show()

+-------+----------+-----+
|   name|department|score|
+-------+----------+-----+
|    Ali|        CS|   88|
|   Mona|        CS|   95|
|Youssef|        IT|   72|
|   Nour|        IT|   81|
|   Sara|        AI|   91|
|   Omar|        AI|   85|
+-------+----------+-----+



## Section 2 — Window Function Exercises

### Exercise 1 — Running total of scores ordered by student_id

In [23]:
w = Window.orderBy("student_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = students.withColumn("running_total", F.sum("score").over(w))
result.show()

+----------+-------+-----+-----+-------------+
|student_id|   name|major|score|running_total|
+----------+-------+-----+-----+-------------+
|       101|    Ali|   CS|   88|           88|
|       102|   Mona|   CS|   95|          183|
|       103|Youssef|   IT|   72|          255|
|       104|   Nour|   IT|   81|          336|
|       105|   Sara|   AI|   91|          427|
|       106|   Omar|   AI|   85|          512|
+----------+-------+-----+-----+-------------+



### Exercise 2 — above_major_avg / below_major_avg

In [24]:
w_major = Window.partitionBy("major")
result = (students
    .withColumn("major_avg", F.avg("score").over(w_major))
    .withColumn("vs_major_avg",
        F.when(F.col("score") >= F.col("major_avg"), "above_major_avg")
         .otherwise("below_major_avg"))
    .drop("major_avg")
)
result.show()

+----------+-------+-----+-----+---------------+
|student_id|   name|major|score|   vs_major_avg|
+----------+-------+-----+-----+---------------+
|       105|   Sara|   AI|   91|above_major_avg|
|       106|   Omar|   AI|   85|below_major_avg|
|       101|    Ali|   CS|   88|below_major_avg|
|       102|   Mona|   CS|   95|above_major_avg|
|       103|Youssef|   IT|   72|below_major_avg|
|       104|   Nour|   IT|   81|above_major_avg|
+----------+-------+-----+-----+---------------+



### Exercise 3 — Top 2 students per major

In [25]:
w_rank = Window.partitionBy("major").orderBy(F.col("score").desc())
result = (students
    .withColumn("rank", F.rank().over(w_rank))
    .filter(F.col("rank") <= 2)
)
result.show()

+----------+-------+-----+-----+----+
|student_id|   name|major|score|rank|
+----------+-------+-----+-----+----+
|       105|   Sara|   AI|   91|   1|
|       106|   Omar|   AI|   85|   2|
|       102|   Mona|   CS|   95|   1|
|       101|    Ali|   CS|   88|   2|
|       104|   Nour|   IT|   81|   1|
|       103|Youssef|   IT|   72|   2|
+----------+-------+-----+-----+----+



## Section 3 — String Function Exercises

In [26]:
contacts = spark.createDataFrame([
    ("  ali123@example.com ",),
    ("sara_99@gmail.com",),
    ("mona.test@yahoo.com",),
    (" ahmed2025@outlook.com ",)
], ["email"])
contacts.show(truncate=False)

+-----------------------+
|email                  |
+-----------------------+
|  ali123@example.com   |
|sara_99@gmail.com      |
|mona.test@yahoo.com    |
| ahmed2025@outlook.com |
+-----------------------+



### Exercise 1 — Extract username and email provider

In [27]:
result = (contacts
    .withColumn("email",    F.trim(F.lower(F.col("email"))))
    .withColumn("username", F.regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("provider", F.regexp_extract("email", r'@([^.]+)', 1))
)
result.show(truncate=False)

+---------------------+---------+--------+
|email                |username |provider|
+---------------------+---------+--------+
|ali123@example.com   |ali123   |example |
|sara_99@gmail.com    |sara_99  |gmail   |
|mona.test@yahoo.com  |mona.test|yahoo   |
|ahmed2025@outlook.com|ahmed2025|outlook |
+---------------------+---------+--------+



### Exercise 2 — Domain name only (gmail, yahoo, outlook …)

In [28]:
result = (contacts
    .withColumn("email",  F.trim(F.lower(F.col("email"))))
    .withColumn("domain", F.regexp_extract("email", r'@([^.]+)', 1))
)
result.show(truncate=False)

+---------------------+-------+
|email                |domain |
+---------------------+-------+
|ali123@example.com   |example|
|sara_99@gmail.com    |gmail  |
|mona.test@yahoo.com  |yahoo  |
|ahmed2025@outlook.com|outlook|
+---------------------+-------+



### Exercise 3 — Clean email (trim + lowercase)

In [29]:
result = contacts.withColumn("email", F.trim(F.lower(F.col("email"))))
result.show(truncate=False)

+---------------------+
|email                |
+---------------------+
|ali123@example.com   |
|sara_99@gmail.com    |
|mona.test@yahoo.com  |
|ahmed2025@outlook.com|
+---------------------+



### Exercise 4 — Count emails per provider

In [30]:
result = (contacts
    .withColumn("email",    F.trim(F.lower(F.col("email"))))
    .withColumn("provider", F.regexp_extract("email", r'@([^.]+)', 1))
    .groupBy("provider")
    .agg(F.count("*").alias("email_count"))
)
result.show()

+--------+-----------+
|provider|email_count|
+--------+-----------+
|   gmail|          1|
| example|          1|
| outlook|          1|
|   yahoo|          1|
+--------+-----------+



### Exercise 5 — Remove digits from usernames

In [31]:
result = (contacts
    .withColumn("email",             F.trim(F.lower(F.col("email"))))
    .withColumn("username_raw",      F.regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("username_no_digits",F.regexp_replace("username_raw", r'[0-9]', ""))
)
result.select("email", "username_raw", "username_no_digits").show(truncate=False)

+---------------------+------------+------------------+
|email                |username_raw|username_no_digits|
+---------------------+------------+------------------+
|ali123@example.com   |ali123      |ali               |
|sara_99@gmail.com    |sara_99     |sara_             |
|mona.test@yahoo.com  |mona.test   |mona.test         |
|ahmed2025@outlook.com|ahmed2025   |ahmed             |
+---------------------+------------+------------------+



## Section 4 — Date & Time Exercises

In [32]:
events = spark.createDataFrame([
    (1, "2025-01-15"),
    (2, "2025-03-22"),
    (3, "2025-07-04"),
    (4, "2025-10-19"),
    (5, "2025-12-31")
], ["event_id", "event_date"])
events.show()

+--------+----------+
|event_id|event_date|
+--------+----------+
|       1|2025-01-15|
|       2|2025-03-22|
|       3|2025-07-04|
|       4|2025-10-19|
|       5|2025-12-31|
+--------+----------+



### Exercise 1 — Convert event_date to DateType

In [33]:
events = events.withColumn("event_date", F.to_date("event_date", "yyyy-MM-dd"))
events.printSchema()
events.show()

root
 |-- event_id: long (nullable = true)
 |-- event_date: date (nullable = true)

+--------+----------+
|event_id|event_date|
+--------+----------+
|       1|2025-01-15|
|       2|2025-03-22|
|       3|2025-07-04|
|       4|2025-10-19|
|       5|2025-12-31|
+--------+----------+



### Exercise 2 — Extract year, month, day

In [34]:
result = (events
    .withColumn("year",  F.year("event_date"))
    .withColumn("month", F.month("event_date"))
    .withColumn("day",   F.dayofmonth("event_date"))
)
result.show()

+--------+----------+----+-----+---+
|event_id|event_date|year|month|day|
+--------+----------+----+-----+---+
|       1|2025-01-15|2025|    1| 15|
|       2|2025-03-22|2025|    3| 22|
|       3|2025-07-04|2025|    7|  4|
|       4|2025-10-19|2025|   10| 19|
|       5|2025-12-31|2025|   12| 31|
+--------+----------+----+-----+---+



### Exercise 3 — Quarter of each event

In [35]:
result = events.withColumn(
    "quarter",
    F.when(F.month("event_date").between(1, 3),  "Q1")
     .when(F.month("event_date").between(4, 6),  "Q2")
     .when(F.month("event_date").between(7, 9),  "Q3")
     .otherwise("Q4")
)
result.show()

+--------+----------+-------+
|event_id|event_date|quarter|
+--------+----------+-------+
|       1|2025-01-15|     Q1|
|       2|2025-03-22|     Q1|
|       3|2025-07-04|     Q3|
|       4|2025-10-19|     Q4|
|       5|2025-12-31|     Q4|
+--------+----------+-------+



### Exercise 4 — Days remaining until end of event's year

In [36]:
result = events.withColumn(
    "days_until_year_end",
    F.datediff(
        F.to_date(F.concat(F.year("event_date").cast("string"), F.lit("-12-31")), "yyyy-MM-dd"),
        F.col("event_date")
    )
)
result.show()

+--------+----------+-------------------+
|event_id|event_date|days_until_year_end|
+--------+----------+-------------------+
|       1|2025-01-15|                350|
|       2|2025-03-22|                284|
|       3|2025-07-04|                180|
|       4|2025-10-19|                 73|
|       5|2025-12-31|                  0|
+--------+----------+-------------------+



### Exercise 5 — Events in the second half of the year (July–December)

In [37]:
result = events.filter(F.month("event_date") >= 7)
result.show()

+--------+----------+
|event_id|event_date|
+--------+----------+
|       3|2025-07-04|
|       4|2025-10-19|
|       5|2025-12-31|
+--------+----------+



## Section 5 — Array Function Exercises

In [38]:
courses = spark.createDataFrame([
    ("Python",  ["Programming", "Beginner", "Data"]),
    ("Spark",   ["Big Data", "Programming"]),
    ("SQL",     ["Database", "Data"]),
    ("Airflow", ["Workflow", "Automation", "Data"]),
    ("Linux",   ["OS", "Administration"])
], ["course_name", "topics"])
courses.show(truncate=False)

+-----------+-----------------------------+
|course_name|topics                       |
+-----------+-----------------------------+
|Python     |[Programming, Beginner, Data]|
|Spark      |[Big Data, Programming]      |
|SQL        |[Database, Data]             |
|Airflow    |[Workflow, Automation, Data] |
|Linux      |[OS, Administration]         |
+-----------+-----------------------------+



### Exercise 1 — Courses with more than 2 topics

In [39]:
result = courses.filter(F.size("topics") > 2)
result.show(truncate=False)

+-----------+-----------------------------+
|course_name|topics                       |
+-----------+-----------------------------+
|Python     |[Programming, Beginner, Data]|
|Airflow    |[Workflow, Automation, Data] |
+-----------+-----------------------------+



### Exercise 2 — has_data_topic column (True/False)

In [40]:
result = courses.withColumn("has_data_topic", F.array_contains("topics", "Data"))
result.show(truncate=False)

+-----------+-----------------------------+--------------+
|course_name|topics                       |has_data_topic|
+-----------+-----------------------------+--------------+
|Python     |[Programming, Beginner, Data]|true          |
|Spark      |[Big Data, Programming]      |false         |
|SQL        |[Database, Data]             |true          |
|Airflow    |[Workflow, Automation, Data] |true          |
|Linux      |[OS, Administration]         |false         |
+-----------+-----------------------------+--------------+



### Exercise 3 — Count courses per topic using explode()

In [41]:
result = (courses
    .withColumn("topic", F.explode("topics"))
    .groupBy("topic")
    .agg(F.count("course_name").alias("course_count"))
    .orderBy("course_count", ascending=False)
)
result.show()

+--------------+------------+
|         topic|course_count|
+--------------+------------+
|          Data|           3|
|   Programming|           2|
|      Big Data|           1|
|      Beginner|           1|
|Administration|           1|
|      Workflow|           1|
|            OS|           1|
|    Automation|           1|
|      Database|           1|
+--------------+------------+



### Exercise 4 — Courses containing the topic "Programming"

In [42]:
result = courses.filter(F.array_contains("topics", "Programming"))
result.show(truncate=False)

+-----------+-----------------------------+
|course_name|topics                       |
+-----------+-----------------------------+
|Python     |[Programming, Beginner, Data]|
|Spark      |[Big Data, Programming]      |
+-----------+-----------------------------+



### Exercise 5 — Number of topics per course

In [43]:
result = courses.withColumn("topic_count", F.size("topics"))
result.show(truncate=False)

+-----------+-----------------------------+-----------+
|course_name|topics                       |topic_count|
+-----------+-----------------------------+-----------+
|Python     |[Programming, Beginner, Data]|3          |
|Spark      |[Big Data, Programming]      |2          |
|SQL        |[Database, Data]             |2          |
|Airflow    |[Workflow, Automation, Data] |3          |
|Linux      |[OS, Administration]         |2          |
+-----------+-----------------------------+-----------+



## Capstone — Online Learning Platform Analytics

In [44]:
students_cap = spark.createDataFrame([
    (1, "Ali"), (2, "Mona"), (3, "Sara"), (4, "Omar"), (5, "Nour")
], ["student_id", "student_name"])

enrollments = spark.createDataFrame([
    (1, "Python", 88), (1, "SQL", 92),
    (2, "Python", 95), (2, "Spark", 90),
    (3, "SQL",    78), (3, "Spark", 85),
    (4, "Python", 70),
    (5, "Spark",  98)
], ["student_id", "course", "score"])

print("students_cap:"); students_cap.show()
print("enrollments:");  enrollments.show()

students_cap:
+----------+------------+
|student_id|student_name|
+----------+------------+
|         1|         Ali|
|         2|        Mona|
|         3|        Sara|
|         4|        Omar|
|         5|        Nour|
+----------+------------+

enrollments:
+----------+------+-----+
|student_id|course|score|
+----------+------+-----+
|         1|Python|   88|
|         1|   SQL|   92|
|         2|Python|   95|
|         2| Spark|   90|
|         3|   SQL|   78|
|         3| Spark|   85|
|         4|Python|   70|
|         5| Spark|   98|
+----------+------+-----+



### Part 1 — Data Preparation
#### Task 1 — Join students with enrollments

In [45]:
joined = students_cap.join(enrollments, on="student_id", how="inner")
joined.show()

+----------+------------+------+-----+
|student_id|student_name|course|score|
+----------+------------+------+-----+
|         1|         Ali|Python|   88|
|         1|         Ali|   SQL|   92|
|         2|        Mona|Python|   95|
|         2|        Mona| Spark|   90|
|         3|        Sara|   SQL|   78|
|         3|        Sara| Spark|   85|
|         4|        Omar|Python|   70|
|         5|        Nour| Spark|   98|
+----------+------------+------+-----+



#### Task 2 — Schema and data quality

In [46]:
joined.printSchema()
print("Row count:", joined.count())
print("Null counts:")
joined.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in joined.columns]).show()

root
 |-- student_id: long (nullable = true)
 |-- student_name: string (nullable = true)
 |-- course: string (nullable = true)
 |-- score: long (nullable = true)

Row count: 8
Null counts:
+----------+------------+------+-----+
|student_id|student_name|course|score|
+----------+------------+------+-----+
|         0|           0|     0|    0|
+----------+------------+------+-----+



#### Task 3 — Remove duplicate records

In [47]:
joined = joined.dropDuplicates()
print("Row count after dedup:", joined.count())

Row count after dedup: 8


### Part 2 — Aggregations
#### Task 4 — Average score per student

In [48]:
avg_per_student = (joined
    .groupBy("student_id", "student_name")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
avg_per_student.show()

+----------+------------+---------+
|student_id|student_name|avg_score|
+----------+------------+---------+
|         5|        Nour|     98.0|
|         2|        Mona|     92.5|
|         1|         Ali|     90.0|
|         3|        Sara|     81.5|
|         4|        Omar|     70.0|
+----------+------------+---------+



#### Task 5 — Average score per course

In [49]:
avg_per_course = (joined
    .groupBy("course")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
)
avg_per_course.show()

+------+-----------------+
|course|        avg_score|
+------+-----------------+
| Spark|             91.0|
|   SQL|             85.0|
|Python|84.33333333333333|
+------+-----------------+



#### Task 6 — Highest score per course

In [50]:
highest_per_course = (joined
    .groupBy("course")
    .agg(F.max("score").alias("highest_score"))
)
highest_per_course.show()

+------+-------------+
|course|highest_score|
+------+-------------+
|   SQL|           92|
|Python|           95|
| Spark|           98|
+------+-------------+



#### Task 7 — Student with the highest average score overall

In [51]:
result = (joined
    .groupBy("student_id", "student_name")
    .agg(F.avg("score").alias("avg_score"))
    .orderBy("avg_score", ascending=False)
    .limit(1)
)
result.show()

+----------+------------+---------+
|student_id|student_name|avg_score|
+----------+------------+---------+
|         5|        Nour|     98.0|
+----------+------------+---------+



### Part 3 — Window Functions
#### Task 8 — Rank students within each course

In [52]:
w_course = Window.partitionBy("course").orderBy(F.col("score").desc())
ranked = joined.withColumn("course_rank", F.rank().over(w_course))
ranked.show()

+----------+------------+------+-----+-----------+
|student_id|student_name|course|score|course_rank|
+----------+------------+------+-----+-----------+
|         2|        Mona|Python|   95|          1|
|         1|         Ali|Python|   88|          2|
|         4|        Omar|Python|   70|          3|
|         1|         Ali|   SQL|   92|          1|
|         3|        Sara|   SQL|   78|          2|
|         5|        Nour| Spark|   98|          1|
|         2|        Mona| Spark|   90|          2|
|         3|        Sara| Spark|   85|          3|
+----------+------------+------+-----+-----------+



#### Task 9 — Running total of scores within each course

In [53]:
w_running = Window.partitionBy("course").orderBy("student_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = joined.withColumn("running_total", F.sum("score").over(w_running))
result.show()

+----------+------------+------+-----+-------------+
|student_id|student_name|course|score|running_total|
+----------+------------+------+-----+-------------+
|         1|         Ali|Python|   88|           88|
|         2|        Mona|Python|   95|          183|
|         4|        Omar|Python|   70|          253|
|         1|         Ali|   SQL|   92|           92|
|         3|        Sara|   SQL|   78|          170|
|         2|        Mona| Spark|   90|           90|
|         3|        Sara| Spark|   85|          175|
|         5|        Nour| Spark|   98|          273|
+----------+------------+------+-----+-------------+



#### Task 10 — Difference between student score and course average

In [54]:
w_avg = Window.partitionBy("course")
result = (joined
    .withColumn("course_avg", F.avg("score").over(w_avg))
    .withColumn("diff_from_avg", F.col("score") - F.col("course_avg"))
)
result.show()

+----------+------------+------+-----+-----------------+-------------------+
|student_id|student_name|course|score|       course_avg|      diff_from_avg|
+----------+------------+------+-----+-----------------+-------------------+
|         1|         Ali|Python|   88|84.33333333333333| 3.6666666666666714|
|         2|        Mona|Python|   95|84.33333333333333| 10.666666666666671|
|         4|        Omar|Python|   70|84.33333333333333|-14.333333333333329|
|         1|         Ali|   SQL|   92|             85.0|                7.0|
|         3|        Sara|   SQL|   78|             85.0|               -7.0|
|         2|        Mona| Spark|   90|             91.0|               -1.0|
|         3|        Sara| Spark|   85|             91.0|               -6.0|
|         5|        Nour| Spark|   98|             91.0|                7.0|
+----------+------------+------+-----+-----------------+-------------------+



### Part 4 — Business Rules
#### Task 11 — Performance category

In [55]:
with_perf = joined.withColumn(
    "performance",
    F.when(F.col("score") >= 90, "Excellent")
     .when(F.col("score") >= 80, "Good")
     .otherwise("Needs Improvement")
)
with_perf.show()

+----------+------------+------+-----+-----------------+
|student_id|student_name|course|score|      performance|
+----------+------------+------+-----+-----------------+
|         1|         Ali|Python|   88|             Good|
|         1|         Ali|   SQL|   92|        Excellent|
|         2|        Mona|Python|   95|        Excellent|
|         2|        Mona| Spark|   90|        Excellent|
|         3|        Sara|   SQL|   78|Needs Improvement|
|         3|        Sara| Spark|   85|             Good|
|         4|        Omar|Python|   70|Needs Improvement|
|         5|        Nour| Spark|   98|        Excellent|
+----------+------------+------+-----+-----------------+



#### Task 12 — Count students per performance category

In [56]:
result = (with_perf
    .groupBy("performance")
    .agg(F.count("student_id").alias("student_count"))
    .orderBy("student_count", ascending=False)
)
result.show()

+-----------------+-------------+
|      performance|student_count|
+-----------------+-------------+
|        Excellent|            4|
|Needs Improvement|            2|
|             Good|            2|
+-----------------+-------------+



### Part 5 — Final Report

In [57]:
w_final = Window.partitionBy("course").orderBy(F.col("score").desc())

final_report = (joined
    .withColumn("performance",
        F.when(F.col("score") >= 90, "Excellent")
         .when(F.col("score") >= 80, "Good")
         .otherwise("Needs Improvement"))
    .withColumn("course_rank", F.rank().over(w_final))
    .select("student_name", "course", "score", "course_rank", "performance")
    .orderBy("course", "course_rank")
)
final_report.show()

+------------+------+-----+-----------+-----------------+
|student_name|course|score|course_rank|      performance|
+------------+------+-----+-----------+-----------------+
|        Mona|Python|   95|          1|        Excellent|
|         Ali|Python|   88|          2|             Good|
|        Omar|Python|   70|          3|Needs Improvement|
|         Ali|   SQL|   92|          1|        Excellent|
|        Sara|   SQL|   78|          2|Needs Improvement|
|        Nour| Spark|   98|          1|        Excellent|
|        Mona| Spark|   90|          2|        Excellent|
|        Sara| Spark|   85|          3|             Good|
+------------+------+-----+-----------+-----------------+

